# **Problem Statement**

## **Business Context**

GreenBasket Grocers is an online grocery delivery service headquartered in Austin, Texas, operating across 14 major metropolitan areas in the United States, including Austin, Dallas, Houston, Phoenix, Denver, Atlanta, Charlotte, Nashville, Tampa, Orlando, Columbus, Indianapolis, Kansas City, and Salt Lake City. The company runs a network of 140 dark stores supported by around 3,200 gig-based delivery partners, and fulfills close to 85,000 orders per day. It offers fresh produce, dairy, bakery, and packaged goods through scheduled 2-hour delivery windows, with an average order value of 42 dollars and a customer base of over 4 million registered users. GreenBasket has positioned itself as a convenient, reliable alternative to traditional grocery shopping, competing on delivery speed, product freshness, and slot availability.

A recurring operational challenge for the business is failed deliveries. On any given day, roughly 8 to 10 percent of orders either arrive late, get refused by the customer at the door, or come back undelivered due to reasons such as incorrect addresses, customers being unavailable, damaged perishables, or payment issues at the point of delivery. Each failed delivery costs the company between 12 and 28 dollars when accounting for wasted perishables, delivery partner payouts, reverse logistics, and customer service overhead. At the current failure rate, this translates to daily losses of nearly 150,000 to 200,000 dollars, in addition to the harder-to-measure damage to customer trust and repeat purchase behavior. Today, the operations team treats all orders the same way at dispatch, with no reliable method to identify which orders are at high risk of failing before the delivery is attempted.

If GreenBasket can accurately identify high-risk orders at the time of order placement, the operations team can take targeted actions such as confirmation calls, prepaid payment nudges, priority routing, or slot reassignment for those specific orders. Even a 25 percent reduction in the failure rate would save the company close to 18 million dollars annually, meaningfully cut down on perishable wastage, and improve the on-time delivery experience for millions of customers. It would also allow the company to allocate its customer service and dispatch resources more efficiently, focusing effort where it actually moves the needle rather than spreading it thinly across every order.

## **Objective**

To build a classification model that can predict whether an order is likely to fail delivery at the time of order placement, using features such as customer order history, past failure behavior, delivery address characteristics, order value, payment mode, item mix, chosen time slot, and delivery partner load, enabling the operations team to proactively intervene on high-risk orders through confirmation calls, prepaid nudges, priority routing, or slot reassignment, and helping the company reduce perishable wastage, cut delivery costs, and improve overall customer experience.

## **Data Dictionary**

The dataset, `greenbasket_deliveries.csv`, contains 45,000 order-level records spanning a 45-day operational window across the 14 metropolitan markets served by GreenBasket.

### Identifiers & Metadata

| Column | Data Type | Description |
| --- | --- | --- |
| order_id | object | Unique identifier assigned to each order at the time of placement. |
| order_timestamp | datetime | Date and time when the order was placed. |
| customer_id | object | Unique identifier for the customer placing the order. |

### Customer History

| Column | Data Type | Description |
| --- | --- | --- |
| customer_tenure_days | int | Number of days since the customer registered on the platform. |
| past_order_count | int | Total number of prior orders placed by the customer before this one. |
| past_failure_count | int | Total number of prior orders that failed delivery for this customer. |
| past_failure_rate | float | Historical failure rate for the customer, smoothed with a Laplace prior. |
| avg_past_order_value_usd | float | Average value of the customer's prior orders in USD. |
| days_since_last_order | float | Number of days since the customer's most recent prior order. |

### Delivery Address

| Column | Data Type | Description |
| --- | --- | --- |
| delivery_city | object | City where the order is to be delivered. |
| delivery_pin_code | object | Postal code (ZIP) of the delivery location. |
| is_new_address | int | Flag indicating whether this is the first time delivering to this address for the customer. |
| address_type | object | Type of address (apartment, house, office, other). |
| floor_number | int | Floor number, applicable only for apartment deliveries. |
| distance_from_darkstore_km | float | Road distance from the fulfillment dark store to the delivery address. |
| pin_code_failure_rate | float | Historical failure rate at the delivery pin code, smoothed with a Laplace prior. |

### Order Details

| Column | Data Type | Description |
| --- | --- | --- |
| order_value_usd | float | Total value of the order in USD. |
| num_items | int | Total number of items in the order. |
| num_perishable_items | int | Number of perishable items in the order. |
| num_frozen_items | int | Number of frozen items in the order. |
| has_alcohol | int | Flag indicating whether the order contains alcohol, requiring age verification. |
| payment_mode | object | Payment method (prepaid_card, prepaid_wallet, cash_on_delivery). |
| discount_applied_pct | float | Percentage discount applied to the order. |

### Slot & Time

| Column | Data Type | Description |
| --- | --- | --- |
| delivery_slot | object | Chosen delivery window (morning, afternoon, evening, night). |
| day_of_week | object | Day of the week the order was placed. |
| is_weekend | int | Flag indicating whether the order was placed on a weekend. |
| lead_time_hours | float | Time in hours between order placement and the scheduled delivery slot. |

### Operational

| Column | Data Type | Description |
| --- | --- | --- |
| partner_current_load | float | Number of orders currently assigned to the delivery partner in the same slot. |
| darkstore_utilization_pct | float | Fulfillment capacity utilization of the originating dark store at the time of order. |
| weather_condition | object | Weather at the delivery location (clear, cloudy, rain, storm, snow). |
| temperature_f | float | Outside temperature at the delivery location in degrees Fahrenheit. |

### Target

| Column | Data Type | Description |
| --- | --- | --- |
| delivery_failed | int | Binary target variable indicating whether the delivery failed (1) or was successful (0). |

# **Please read the instructions carefully before starting the project.**

This is a commented Python notebook file in which all the instructions and tasks to be performed are mentioned.

* A significant portion of the code required to conduct the analysis and build and evaluate the predictive models are pre-written in the notebook, and only need to be executed.
* In certain sections, blanks '\_\_\_\_\_' are provided in the notebook that
needs to be filled appropriately to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* In certain sections, mutliple lines of code are provided and commented out. The appropriate code snippet needs to be uncommented before executing the respective code cells.
* Identify the task to be performed correctly, and only then proceed to fill in the blank or uncomment code snippets.
* Sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in the presentation and submit the same. Any mathematical or computational details that are a graded part of the project can be included in the Appendix section of the presentation.

# **Installing and Importing the Necessary Libraries**

In [ ]:
!pip install pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 tensorflow==2.20.0 keras==3.13.2 xgboost==3.2.0 seaborn==0.13.2 matplotlib==3.10.0 -q

**Note**:
- After running the above cell, kindly restart the notebook kernel (for VS Code) or runtime (for Google Colab), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Standard libraries for tracking execution time and vector/matrix operations
import time
import numpy as np

# Data manipulation and visualization libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tree-based and ensemble machine learning classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Utilities for handling class imbalance, model evaluation, and metric calculations
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# Deep learning framework and specific layers for building Artificial Neural Networks (ANNs)
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization

# Preprocessing tools for scaling data and tools for model optimization/explainability
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import permutation_importance

# Configuration settings to suppress warnings and format data display outputs
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# **Loading the Data**

In [ ]:
data = pd.read_csv('greenbasket_deliveries.csv')

# **Data Overview**

## Viewing the first and last 5 rows of the dataset

In [ ]:
data.head()

In [ ]:
data.tail()

## Checking the shape of the dataset

In [ ]:
data.shape

## Checking the attribute types

In [ ]:
data.info()

## Checking the statistical summary

In [ ]:
data.describe().T

## Checking for missing values

In [ ]:
data.isnull().sum()

## Checking for duplicate values

In [ ]:
data.duplicated().sum()

# **Data Preprocessing**

Before beginning exploratory analysis, we convert the `order_timestamp` column to a proper datetime type and extract two temporal features that will be used during EDA and modeling.

In [ ]:
data["order_timestamp"] = pd.to_datetime(data["order_timestamp"])

In [ ]:
data["dayofweek"] = data["order_timestamp"].dt.dayofweek

In [ ]:
data["hour"] = data["order_timestamp"].dt.hour

# **Exploratory Data Analysis**

## Utility Functions

In [ ]:
def histogram(data_df, col, title='Histogram', xlabel=None, ylabel='Frequency'):
    plt.figure(figsize=(9, 3.6))
    sns.histplot(data_df[col], bins=50, kde=True)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else col)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

def barchart(data_df, x_col, y_col=None, title='Bar Chart', xlabel=None, ylabel=None, rot_degrees=30):
    plt.figure(figsize=(10, 6))
    if y_col:
        sns.barplot(x=x_col, y=y_col, data=data_df)
    else:
        order = data_df[x_col].value_counts().index
        sns.countplot(data=data_df, x=x_col, order=order)

    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else ('Count' if not y_col else y_col))
    plt.xticks(rotation=rot_degrees, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def boxplot(data_df, x_col, y_col, title='Box Plot', xlabel=None, ylabel=None):
    plt.figure(figsize=(6, 3.8))
    sns.boxplot(data=data_df, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.tight_layout()
    plt.show()

## Univariate Analysis

### `delivery_failed`

In [ ]:
barchart(data, 'delivery_failed', title='Target Distribution: Delivery Outcome', xlabel='delivery_failed (0 = successful, 1 = failed)')

### `past_failure_rate`

In [ ]:
histogram(data,'past_failure_rate', title='Distribution of past_failure_rate', xlabel='past_failure_rate')

### `pin_code_failure_rate`

In [ ]:
# Fill in the blank with the column name to analyze
histogram(data, '____', title='Distribution of pin_code_failure_rate', xlabel='pin_code_failure_rate')

### `distance_from_darkstore_km`

In [ ]:
# Fill in the blank with the column name to analyze
histogram(data, '____', title='Distribution of distance_from_darkstore_km', xlabel='distance_from_darkstore_km')

### `order_value_usd`

In [ ]:
# Fill in the blank with the column name to analyze
histogram(data, '____', title='Distribution of order_value_usd', xlabel='order_value_usd')

## Bivariate Analysis

### `delivery_failed` vs `hour`

In [ ]:
failure_rate_by_hour_df = data.groupby('hour')['delivery_failed'].mean().reset_index()
failure_rate_by_hour_df['failure_rate_pct'] = failure_rate_by_hour_df['delivery_failed'] * 100

barchart(data_df=failure_rate_by_hour_df, x_col='hour', y_col='failure_rate_pct',
         title='Failure Rate by Hour of Day',
         xlabel='Hour of Day', ylabel='Failure Rate (%)', rot_degrees=0)

### `delivery_failed` vs `payment_mode`

In [ ]:
# Fill in the blank with the column name to group the data by
failure_rate_by_payment_df = data.groupby('____')['delivery_failed'].mean().reset_index()
failure_rate_by_payment_df['failure_rate_pct'] = failure_rate_by_payment_df['delivery_failed'] * 100

barchart(data_df=failure_rate_by_payment_df, x_col='payment_mode', y_col='failure_rate_pct',
         title='Failure Rate by Payment Mode',
         xlabel='Payment Mode', ylabel='Failure Rate (%)', rot_degrees=0)

### `past_failure_rate` vs `delivery_failed`

In [ ]:
boxplot(data, x_col='delivery_failed', y_col='past_failure_rate',
        title='Past Failure Rate vs Delivery Outcome',
        xlabel='delivery_failed (0 = successful, 1 = failed)',
        ylabel='past_failure_rate')

### `distance_from_darkstore_km` vs `delivery_failed`

In [ ]:
# Fill in the blank with the column name to analyze against the target
boxplot(data, x_col='delivery_failed', y_col='____',
        title='Distance from Dark Store vs Delivery Outcome',
        xlabel='delivery_failed (0 = successful, 1 = failed)',
        ylabel='distance_from_darkstore_km (km)')

## Multivariate Analysis

### Correlation Analysis

In [ ]:
numerical_cols = ['customer_tenure_days', 'past_order_count', 'past_failure_count', 'past_failure_rate',
                  'avg_past_order_value_usd', 'days_since_last_order', 'is_new_address', 'floor_number',
                  'distance_from_darkstore_km', 'pin_code_failure_rate', 'order_value_usd', 'num_items',
                  'num_perishable_items', 'num_frozen_items', 'has_alcohol', 'discount_applied_pct',
                  'is_weekend', 'lead_time_hours', 'partner_current_load', 'darkstore_utilization_pct',
                  'temperature_f', 'hour', 'dayofweek', 'delivery_failed']

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(data[numerical_cols].corr(), annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# **Data Preprocessing**

## Splitting the data into train, validation, and test sets

In [ ]:
data = data.sort_values(by='order_timestamp').reset_index(drop=True)

In [ ]:
n = len(data)

In [ ]:
# Uncomment the below train and validation indices to create a 70% train, 15% validation, and 15% test split.
# i_train = int(n * 0.70)
# i_val = int(n * 0.85)

# Uncomment the below train and validation indices to create a 80% train, 10% validation, and 10% test split.
# i_train = int(n * 0.80)
# i_val = int(n * 0.90)

# Uncomment the below train and validation indices to create a 90% train, 5% validation, and 5% test split.
# i_train = int(n * 0.90)
# i_val = int(n * 0.95)

In [ ]:
data_train = data.iloc[:i_train]
data_val = data.iloc[i_train:i_val]
data_test = data.iloc[i_val:]

drop_cols = ['delivery_failed', 'order_timestamp', 'order_id', 'customer_id', 'delivery_pin_code']

X_train = data_train.drop(columns=drop_cols)
y_train = data_train['delivery_failed']

X_valid = data_val.drop(columns=drop_cols)
y_valid = data_val['delivery_failed']

X_test = data_test.drop(columns=drop_cols)
y_test = data_test['delivery_failed']

In [ ]:
print(f'Train split failure rate: {y_train.mean()*100:.2f}%')
print(f'Validation split failure rate: {y_valid.mean()*100:.2f}%')
print(f'Test split failure rate: {y_test.mean()*100:.2f}%')

The categorical columns need to be one-hot encoded before they can be fed to the tree ensembles and the neural network. We fit the encoding on training data and align the validation and test sets to the same set of columns.

In [ ]:
categorical_cols = ['delivery_city', 'address_type', 'payment_mode', 'delivery_slot', 'day_of_week', 'weather_condition']

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_valid = pd.get_dummies(X_valid, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Convert any boolean dummy columns to int for downstream libraries
bool_cols = X_train.select_dtypes(include=['bool']).columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_valid[bool_cols] = X_valid[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

print('Feature matrix shape after encoding:', X_train.shape)

## Missing Value Treatment

### Display the percentage of missing values

In [ ]:
print(((X_train.isnull().sum() / len(X_train) * 100).round(2)).loc[lambda x: x > 0])
print(((X_valid.isnull().sum() / len(X_valid) * 100).round(2)).loc[lambda x: x > 0])
print(((X_test.isnull().sum() / len(X_test) * 100).round(2)).loc[lambda x: x > 0])

### `avg_past_order_value_usd`

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# avg_val_train_stat = X_train['avg_past_order_value_usd'].mean()    # to impute using mean
# avg_val_train_stat = X_train['avg_past_order_value_usd'].median()    # to impute using median
# avg_val_train_stat = X_train['avg_past_order_value_usd'].mode()[0]    # to impute using mode

In [ ]:
X_train['avg_past_order_value_usd'] = X_train['avg_past_order_value_usd'].fillna(avg_val_train_stat)
X_valid['avg_past_order_value_usd'] = X_valid['avg_past_order_value_usd'].fillna(avg_val_train_stat)
X_test['avg_past_order_value_usd']  = X_test['avg_past_order_value_usd'].fillna(avg_val_train_stat)

### `days_since_last_order`

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# days_last_train_stat = X_train['days_since_last_order'].mean()    # to impute using mean
# days_last_train_stat = X_train['days_since_last_order'].median()    # to impute using median
# days_last_train_stat = X_train['days_since_last_order'].mode()[0]    # to impute using mode

In [ ]:
X_train['days_since_last_order'] = X_train['days_since_last_order'].fillna(days_last_train_stat)
X_valid['days_since_last_order'] = X_valid['days_since_last_order'].fillna(days_last_train_stat)
X_test['days_since_last_order']  = X_test['days_since_last_order'].fillna(days_last_train_stat)

### `partner_current_load`

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# partner_load_train_stat = X_train['partner_current_load'].mean()    # to impute using mean
# partner_load_train_stat = X_train['partner_current_load'].median()    # to impute using median
# partner_load_train_stat = X_train['partner_current_load'].mode()[0]    # to impute using mode

In [ ]:
X_train['partner_current_load'] = X_train['partner_current_load'].fillna(partner_load_train_stat)
X_valid['partner_current_load'] = X_valid['partner_current_load'].fillna(partner_load_train_stat)
X_test['partner_current_load']  = X_test['partner_current_load'].fillna(partner_load_train_stat)

### `darkstore_utilization_pct`

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# darkstore_util_train_stat = X_train['darkstore_utilization_pct'].mean()    # to impute using mean
# darkstore_util_train_stat = X_train['darkstore_utilization_pct'].median()    # to impute using median
# darkstore_util_train_stat = X_train['darkstore_utilization_pct'].mode()[0]    # to impute using mode

In [ ]:
X_train['darkstore_utilization_pct'] = X_train['darkstore_utilization_pct'].fillna(darkstore_util_train_stat)
X_valid['darkstore_utilization_pct'] = X_valid['darkstore_utilization_pct'].fillna(darkstore_util_train_stat)
X_test['darkstore_utilization_pct']  = X_test['darkstore_utilization_pct'].fillna(darkstore_util_train_stat)

### `temperature_f`

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# temp_train_stat = X_train['temperature_f'].mean()    # to impute using mean
# temp_train_stat = X_train['temperature_f'].median()    # to impute using median
# temp_train_stat = X_train['temperature_f'].mode()[0]    # to impute using mode

In [ ]:
X_train['temperature_f'] = X_train['temperature_f'].fillna(temp_train_stat)
X_valid['temperature_f'] = X_valid['temperature_f'].fillna(temp_train_stat)
X_test['temperature_f']  = X_test['temperature_f'].fillna(temp_train_stat)

# **Model Building**

## Model Evaluation Criterion

In [ ]:
# Uncomment one of the following evaluation metrics

# metric_of_choice = 'accuracy'
# metric_of_choice = 'precision'
# metric_of_choice = 'recall'
# metric_of_choice = 'f1'

## Utility Functions

In [ ]:
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier (sklearn or tensorflow)
    predictors: independent variables
    target: dependent variable
    """

    pred_raw = model.predict(predictors)

    if np.issubdtype(pred_raw.dtype, np.floating) and not np.all(np.isin(pred_raw, [0.0, 1.0])):
        pred = (pred_raw > 0.5).astype(int)
    else:
        pred = pred_raw

    pred = np.array(pred).flatten()

    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    data_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )

    return data_perf

In [ ]:
def plot_confusion_matrix(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred_raw = model.predict(predictors)

    if np.issubdtype(y_pred_raw.dtype, np.floating) and np.any((y_pred_raw > 0) & (y_pred_raw < 1)):
        y_pred = (y_pred_raw > 0.5).astype(int)
    else:
        y_pred = y_pred_raw

    y_pred = np.array(y_pred).flatten()

    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show()

## **Decision Tree Classifier**

In [ ]:
# Set the random state for reproducibility.
# Choose an integer between 0 and 100.
# Using the same random state ensures the model produces the same results each time it is trained.
RS = __

In [ ]:
dt_class_model = DecisionTreeClassifier(class_weight='balanced', random_state=RS)

In [ ]:
dt_class_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(dt_class_model, X_train, y_train)

In [ ]:
decision_tree_class_weight_perf_train = model_performance_classification(
    dt_class_model, X_train, y_train
)
decision_tree_class_weight_perf_train

In [ ]:
plot_confusion_matrix(dt_class_model, X_valid, y_valid)

In [ ]:
decision_tree_class_weight_perf_valid = model_performance_classification(
    dt_class_model, X_valid, y_valid
)
decision_tree_class_weight_perf_valid

## **Random Forest Classifier**

In [ ]:
rf_class_model = RandomForestClassifier(class_weight='balanced', random_state=RS)

In [ ]:
rf_class_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(rf_class_model, X_train, y_train)

In [ ]:
random_forest_class_weight_perf_train = model_performance_classification(
    rf_class_model, X_train, y_train
)
random_forest_class_weight_perf_train

In [ ]:
plot_confusion_matrix(rf_class_model, X_valid, y_valid)

In [ ]:
random_forest_class_weight_perf_valid = model_performance_classification(
    rf_class_model, X_valid, y_valid
)
random_forest_class_weight_perf_valid

## **Gradient Boosting Classifier**

In [ ]:
gb_model = GradientBoostingClassifier(random_state=RS)

In [ ]:
gb_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(gb_model, X_train, y_train)

In [ ]:
gradient_boosting_weight_perf_train = model_performance_classification(
    gb_model, X_train, y_train
)
gradient_boosting_weight_perf_train

In [ ]:
plot_confusion_matrix(gb_model, X_valid, y_valid)

In [ ]:
gradient_boosting_weight_perf_valid = model_performance_classification(
    gb_model, X_valid, y_valid
)
gradient_boosting_weight_perf_valid

## **XGBoost**

In [ ]:
neg_count = y_train.value_counts()[0]
pos_count = y_train.value_counts()[1]
scale_pos_weight_value = neg_count / pos_count

print(f"Negative Samples: {neg_count}")
print(f"Positive Samples: {pos_count}")
print(f"Scale Pos Weight (neg/pos): {scale_pos_weight_value:.2f}")

In [ ]:
xgb_class_model = XGBClassifier(scale_pos_weight=scale_pos_weight_value, random_state=RS, eval_metric='logloss')

In [ ]:
xgb_class_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(xgb_class_model, X_train, y_train)

In [ ]:
xgb_class_weight_perf_train = model_performance_classification(
    xgb_class_model, X_train, y_train
)
xgb_class_weight_perf_train

In [ ]:
plot_confusion_matrix(xgb_class_model, X_valid, y_valid)

In [ ]:
xgb_class_weight_perf_valid = model_performance_classification(
    xgb_class_model, X_valid, y_valid
)
xgb_class_weight_perf_valid

## **Neural Networks(ANN)**

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Choose the number of neurons for the first hidden layer between 8 and 128 integer value only
# Fewer neurons (8–32) create a simpler model that trains faster and is less likely to overfit.
# More neurons (64–128) allow the model to learn more complex patterns but increase training time and the risk of overfitting.

# Choose the activation function for the first hidden layer.
# Common choices: 'relu', 'tanh', or 'sigmoid'.

model = Sequential()
model.add(Dense(_____, activation='_____', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
optimizer = 'sgd'
loss = 'binary_crossentropy'
model.compile(loss=loss, optimizer=optimizer, metrics=[tf.keras.metrics.Recall(), tf.keras.metrics.Precision(), tf.keras.metrics.BinaryAccuracy()])

In [ ]:
# Choose the number of training epochs between 10 and 100.
# Fewer epochs (10–30) train faster but may underfit the data.
# More epochs (50–100) allow the model to learn more patterns but may increase the risk of overfitting.

epochs = __

# Choose the batch size between 16 and 128.
# Smaller batch sizes (16–32) can improve generalization but increase training time.
# Larger batch sizes (64–128) train faster but may require more memory and sometimes reduce model generalization.

batch_size = __

In [ ]:
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weights[i] for i in range(len(weights))}

In [ ]:
start = time.time()
history = model.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled, y_valid),
                    class_weight=class_weight_dict, batch_size=batch_size, epochs=epochs, verbose=2)
end = time.time()

In [ ]:
print("Time taken in seconds:", round(end - start, 2))

In [ ]:
plot_confusion_matrix(model, X_train_scaled, y_train)

In [ ]:
ann_perf_train = model_performance_classification(
    model, X_train_scaled, y_train
)
print("ANN Baseline Training Performance:")
ann_perf_train

In [ ]:
plot_confusion_matrix(model, X_valid_scaled, y_valid)

In [ ]:
ann_perf_valid = model_performance_classification(
    model, X_valid_scaled, y_valid
)
print("ANN Baseline Validation Performance:")
ann_perf_valid

## **Baseline Model Performance Comparison**

### Training performance comparison

In [ ]:
print("Training performance comparison:")

all_models_train_comp_data = pd.concat([
    decision_tree_class_weight_perf_train.T.rename(columns={0: 'Decision Tree'}),
    random_forest_class_weight_perf_train.T.rename(columns={0: 'Random Forest'}),
    gradient_boosting_weight_perf_train.T.rename(columns={0: 'Gradient Boosting'}),
    xgb_class_weight_perf_train.T.rename(columns={0: 'XGBoost'}),
    ann_perf_train.T.rename(columns={0: 'Neural Network'})
], axis=1)
all_models_train_comp_data

### Validation performance comparison

In [ ]:
print("Validation performance comparison:")

all_models_valid_comp_data = pd.concat([
    decision_tree_class_weight_perf_valid.T.rename(columns={0: 'Decision Tree'}),
    random_forest_class_weight_perf_valid.T.rename(columns={0: 'Random Forest'}),
    gradient_boosting_weight_perf_valid.T.rename(columns={0: 'Gradient Boosting'}),
    xgb_class_weight_perf_valid.T.rename(columns={0: 'XGBoost'}),
    ann_perf_valid.T.rename(columns={0: 'Neural Network'})
], axis=1)
all_models_valid_comp_data

# **Hyperparameter Tuning**

## XGBoost

In [ ]:
param_grid = {
    'n_estimators': [50, ____ , ____ , ____],          # Number of trees. Smaller values (50–100) train faster but may underfit. Larger values (100–300) improve learning but increase training time.
    'learning_rate': [0.01, ____ , ____ , ____],       # Step size for learning. Smaller values (0.01–0.05) learn gradually and need more trees. Larger values (0.1–0.3) learn faster but may overshoot.
    'max_depth': [3, ____ , ____ , ____],              # Maximum tree depth. Smaller values (3–5) reduce overfitting. Larger values (6–10) capture more complex patterns but may overfit.
    'min_child_weight': [1, ____ , ____],       # Minimum weight required for a split. Smaller values (1–3) create more complex trees. Larger values (5–10) make the model more conservative.
    'gamma': [0, 0.1, ____ , ____],             # Minimum loss reduction for a split. Smaller values (0–0.1) allow more splits. Larger values (0.3–1) reduce unnecessary splits.
    'subsample': [0.7, ____ , ____],            # Fraction of training samples per tree. Smaller values (0.6–0.8) improve generalization. Larger values (0.9–1.0) use more data but may overfit.
    'colsample_bytree': [0.4, ____ , ____],     # Fraction of features per tree. Smaller values (0.4–0.7) increase tree diversity. Larger values (0.8–1.0) use more features and may improve fit.
    'reg_alpha': [0, 0.001, ____ , ____],       # L1 regularization. Smaller values (0–0.001) apply little regularization. Larger values (0.01–0.1) reduce overfitting by encouraging sparsity.
    'reg_lambda': [0.1, ____ , ____]            # L2 regularization. Smaller values (0.1–1) apply light regularization. Larger values (5–10) provide stronger regularization.
}

In [ ]:
# Choose the number of random hyperparameter combinations to evaluate.
# Smaller values (10–20) complete faster but explore fewer parameter combinations.
# Larger values (30–50) explore a wider search space and may find better hyperparameters but increase tuning time.

n_iter = ___

xgb_randomized_cv = RandomizedSearchCV(estimator=xgb_class_model, param_distributions=param_grid,
                                       n_iter=n_iter, n_jobs=-1, scoring=metric_of_choice, cv=3, random_state=1)

xgb_randomized_cv.fit(X_train, y_train)

print("Best parameters: {}".format(xgb_randomized_cv.best_params_))
print("Best CV score:   {:.4f}".format(xgb_randomized_cv.best_score_))

In [ ]:
tuned_xgb = xgb_randomized_cv.best_estimator_
tuned_xgb.fit(X_train, y_train)

In [ ]:
xgb_tuned_perf_train = model_performance_classification(
    tuned_xgb, X_train, y_train
)
xgb_tuned_perf_train

In [ ]:
xgb_tuned_perf_valid = model_performance_classification(
    tuned_xgb, X_valid, y_valid
)
xgb_tuned_perf_valid

## Neural Networks

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
model1 = Sequential()

# Choose the number of neurons for the first hidden layer between 32 and 256.
# Fewer neurons (32–64) create a simpler model that trains faster and is less likely to overfit.
# More neurons (128–256) can learn more complex patterns but increase training time and the risk of overfitting.
#
# Choose the activation function for the first hidden layer.
# Common choices are 'relu', 'tanh', or 'sigmoid'.
model1.add(Dense(___, activation='___', input_shape=(X_train_scaled.shape[1],)))

model1.add(BatchNormalization())

# Choose the dropout rate between 0.2 and 0.5.
# Lower values (0.2–0.3) retain more information but provide less regularization.
# Higher values (0.4–0.5) reduce overfitting more aggressively but may lead to underfitting if too high.
model1.add(Dropout(__))

# Choose the number of neurons for the second hidden layer between 16 and 128.
# Fewer neurons (16–32) keep the model simple and reduce overfitting.
# More neurons (64–128) increase model capacity but may overfit on smaller datasets.
#
# Choose the activation function for the second hidden layer.
# Common choices are 'relu', 'tanh', or 'sigmoid'.
model1.add(Dense(___, activation='___'))

model1.add(BatchNormalization())

# Choose the dropout rate between 0.2 and 0.5.
# Lower values (0.2–0.3) retain more information but provide less regularization.
# Higher values (0.4–0.5) reduce overfitting more aggressively but may lead to underfitting if too high.
model1.add(Dropout(___))

model1.add(Dense(16, activation='relu'))
model1.add(Dense(1, activation='sigmoid'))

In [ ]:
model1.summary()

In [ ]:
# Uncomment one of the following optimizers.

# optimizer = 'adam'
# optimizer = 'sgd'

loss = 'binary_crossentropy'
model1.compile(loss=loss, optimizer=optimizer, metrics=[tf.keras.metrics.Recall(), tf.keras.metrics.Precision(), tf.keras.metrics.BinaryAccuracy()])

In [ ]:
# Choose the number of training epochs between 20 and 150.
# Fewer epochs (20–50) train faster but may underfit the data.
# More epochs (75–150) allow the model to learn more complex patterns but may increase the risk of overfitting.

epochs = __

# Choose the batch size between 16 and 128.
# Smaller batch sizes (16–32) update model weights more frequently and may improve generalization but increase training time.
# Larger batch sizes (64–128) train faster and use hardware more efficiently but may require more memory and sometimes reduce generalization.

batch_size = __

In [ ]:
start = time.time()
history = model1.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled, y_valid),
                     class_weight=class_weight_dict, batch_size=batch_size, epochs=epochs, verbose=2)
end = time.time()

In [ ]:
print("Time taken in seconds:", round(end - start, 2))

In [ ]:
plot_confusion_matrix(model1, X_train_scaled, y_train)

In [ ]:
ann_tuned_perf_train = model_performance_classification(
    model1, X_train_scaled, y_train
)
print("ANN Tuned Training Performance:")
ann_tuned_perf_train

In [ ]:
plot_confusion_matrix(model1, X_valid_scaled, y_valid)

In [ ]:
ann_tuned_perf_valid = model_performance_classification(
    model1, X_valid_scaled, y_valid
)
print("ANN Tuned Validation Performance:")
ann_tuned_perf_valid

# **Final Model Selection**

In [ ]:
print("Training performance comparison:")

# Uncomment the chosen tuned models to evaluate whether hyperparameter tuning improved
# the training performance compared to the original models.

all_models_train_comp_data = pd.concat([
    all_models_train_comp_data,
    # xgb_tuned_perf_train.T.rename(columns={0: 'XGBoost (Tuned)'}),
    # ann_tuned_perf_train.T.rename(columns={0: 'Neural Network (Tuned)'})
], axis=1)
all_models_train_comp_data

In [ ]:
print("Validation performance comparison")

# Uncomment the chosen tuned models to evaluate whether hyperparameter tuning improved
# the validation performance compared to the original models.

all_models_valid_comp_data = pd.concat([
    all_models_valid_comp_data,
    # xgb_tuned_perf_valid.T.rename(columns={0: 'XGBoost (Tuned)'}),
    # ann_tuned_perf_valid.T.rename(columns={0: 'Neural Network (Tuned)'})
], axis=1)
all_models_valid_comp_data

In [ ]:
# Uncomment the best-performing baseline or tuned model based on the evaluation metric of your choice.

# Decision Tree
# best_model = dt_class_model

# Random Forest
# best_model = rf_class_model

# Gradient Boosting
# best_model = gb_model

# XGBoost
# best_model = xgb_class_model

# Neural Network
# best_model = model

# XGBoost (Tuned)
# best_model = tuned_xgb

# Neural Network (Tuned)
# best_model = model1

## Feature Importance

In [ ]:
def plot_feature_importances(model, X, y, feature_names=None, color="violet", figsize=(10, 10)):
    """
    Plots feature importances for both traditional ML models and deep learning models.
    """
    if feature_names is None:
        if hasattr(X, 'columns'):
            feature_names = X.columns
        else:
            feature_names = [f"Feature {i}" for i in range(X.shape[1])]

    X_val = X.values if hasattr(X, 'columns') else X

    if hasattr(model, 'feature_importances_'):
        print("Detected tree-based model. Extracting built-in feature importances.")
        importances = model.feature_importances_
        title = "Feature Importances (Tree-based Model)"
        xlabel = "Relative Importance"
    else:
        print("Detected ANN or black-box model. Calculating permutation importance.")
        result = permutation_importance(model, X_val, y, n_repeats=5, random_state=42)
        importances = result.importances_mean
        title = "Feature Importances (Permutation Importance)"
        xlabel = "Mean Performance Drop when Shuffled"

    indices = np.argsort(importances)

    plt.figure(figsize=figsize)
    plt.title(title)
    plt.barh(range(len(indices)), importances[indices], color=color, align="center")
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_feature_importances(best_model, X_train, y_train)

## Final Model Test Performance

In [ ]:
test_perf = model_performance_classification(best_model, X_test, y_test)
test_perf

In [ ]:
plot_confusion_matrix(best_model, X_test, y_test)

# **Business Insights and Recommendations**

## Business Insights

- Add to the presentation

## Recommendations

- Add to the presentation